In [1]:
%load_ext cudf.pandas
import os

# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd

In [2]:
import sys
import os

sys.path.insert(0, os.path.abspath("../.."))
%load_ext ElasticNotebook

Enabled rmm statistics


In [3]:
### cell 0 ###
factor = 2700
course = pd.read_csv(
    "/home/dias-benchmarks/notebooks/aieducation/what-course-are-you-going-to-take/input/course-reviews-university-of-waterloo/course_data_clean.csv"
)
course = pd.concat([course] * factor, ignore_index=True)
course.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 40062600 entries, 0 to 40062599
Data columns (total 10 columns):
 #   Column             Dtype
---  ------             -----
 0   course_code        object
 1   course_title       object
 2   num_ratings        int64
 3   useful             object
 4   easy               object
 5   liked              object
 6   num_reviews        int64
 7   reviews            object
 8   course_rating      object
 9   course_rating_int  float64
dtypes: float64(1), int64(2), object(7)
memory usage: 12.7+ GB


Generating '/var/tmp/nsys-report-809a.qdstrm'
[1/2] [========================100%] all_cells_2025-05-26T01:39:09.nsys-rep
[2/2] [========================100%] all_cells_2025-05-26T01:39:09.sqlite
Generated:
	/home/dias-benchmarks/notebooks/aieducation/what-course-are-you-going-to-take/src/plot_curves/all_cells_2025-05-26T01:39:09.nsys-rep
	/home/dias-benchmarks/notebooks/aieducation/what-course-are-you-going-to-take/src/plot_curves/all_cells_2025-05-26T01:39:09.sqlite
Generating SQLite file /home/dias-benchmarks/notebooks/aieducation/what-course-are-you-going-to-take/src/plot_curves/all_cells_2025-05-26T01:39:09.sqlite from /home/dias-benchmarks/notebooks/aieducation/what-course-are-you-going-to-take/src/plot_curves/all_cells_2025-05-26T01:39:09.nsys-rep
Processing [/home/dias-benchmarks/notebooks/aieducation/what-course-are-you-going-to-take/src/plot_curves/all_cells_2025-05-26T01:39:09.sqlite] with [/opt/nvidia/nsight-systems/2025.3.1/host-linux-x64/reports/nvtx_sum.py]... 

 ** NVTX

In [ ]:
%%time
### cell 1 ###
_ = course.head(10)

In [ ]:
%%time
### cell 2 ###
_ = course.tail(10)

In [ ]:
%%time
### cell 3 ###
_ = course.describe()

In [ ]:
%%time
### cell 4 ###
_ = course.info()

In [ ]:
%%time
### cell 5 ###
course = course.dropna()

In [ ]:
### cell 6 ###
course[["course_unit", "course_num"]] = course["course_code"].str.split(
    " ", expand=True
)

In [ ]:
%%ProfileMem
### cell 7 ###
_ = course[course["num_reviews"] < 10].index

In [ ]:
%%ProfileMem
%%time
### cell 8 ###

def extract_filter_features(df, col_name, threshold):
    """Extracts features for predicting execution time of filtering operation."""
    
    num_rows = len(df)
    dtype_str = str(df[col_name].dtype)  # Get dtype as string
    nan_ratio = df[col_name].isna().mean()  # Compute NaN ratio

    # Compute selectivity: fraction of rows passing the filter
    if df[col_name].dropna().empty:
        target_selectivity = 0.0
    else:
        target_selectivity = (df[col_name] < threshold).mean()

    return {
        'num_rows': num_rows,
        'dtype_str': dtype_str,
        'nan_ratio': nan_ratio,
        'target_selectivity': target_selectivity
    }

_ = extract_filter_features(course, "num_reviews", 10)

In [ ]:
%%time
### cell 9 ###
course.drop(course[course["num_reviews"] < 10].index, inplace=True)

In [ ]:
with rmm.statistics.statistics():
    course[["useful", "easy", "liked", "course_rating"]].nunique()
    print(rmm.statistics.get_statistics().peak_bytes / 1024**3)

In [ ]:
%%time
### cell 10 ###
for i in ["useful", "easy", "liked"]:
    course[i] = course[i].str.replace("%", "")
    course[i] = course[i].astype("int")

In [ ]:
%%time
### cell 11 ###

course.set_index("course_unit", inplace=True)

In [ ]:
%%time
### cell 12 ###

course.drop(["course_title", "reviews", "course_rating"], axis=1, inplace=True)

In [ ]:
%%time
### cell 13 ###
course.info()

In [ ]:
%%time
### cell 14 ###
course_gp = course.groupby("course_unit").mean(numeric_only=True)

In [ ]:
%%time
### cell 15 ###
_ = course_gp.head()

In [ ]:
%%time
### cell 16 ###

# STEFANOS-DISABLE-FOR-MODIN: Modin seems to fail in the LHS, giving an error that "course_rating_mean" is
# not in the DF. Of course, it's valid code because we're creating this column here.
# The problem we cannot just disable this code because follow-up code depends on this column. So, we will create
# the column before the loop so that indexing in the LHS does not fail.

###### ORIGINAL CODE ###########
# for i in course_gp.index:
#     course.loc[i, "course_rating_mean"] = course_gp.loc[i, "course_rating_int"]

###### CODE THAT MODIN CAN RUN ########
course["course_rating_mean"] = None
for i in course_gp.index:
    course.loc[i, "course_rating_mean"] = course_gp.loc[i, "course_rating_int"]

In [ ]:
%%time
### cell 17 ###
course.reset_index(inplace=True)

In [ ]:
%%time
### cell 18 ###

_ = course.groupby("course_unit").mean(numeric_only=True)["course_rating_int"]

In [ ]:
%%time
### cell 19 ###


def extract_features_from_dataframe(df, group_column, agg_function="mean"):
    """Extracts features for predicting the execution time of a groupby operation."""
    n_rows = df.shape[0]
    n_cols = df.shape[1]

    n_groups = df[group_column].nunique()
    group_sizes = df[group_column].value_counts()
    max_group_size = group_sizes.max()

    int_count = df.select_dtypes(include=["int", "int32", "int64"]).shape[1]
    float_count = df.select_dtypes(include=["float", "float32", "float64"]).shape[1]
    str_count = df.select_dtypes(include=["object", "string"]).shape[1]

    aggregation_mapping = {"sum": 0, "mean": 1, "count": 2}
    agg_function_encoded = aggregation_mapping.get(agg_function, 1)  # Default to 'mean'

    return {
        "n_rows": n_rows,
        "n_cols": n_cols,
        "group_range": n_groups,  # Alias for n_groups
        "n_groups": n_groups,
        "max_group_size": max_group_size,
        "int_count": int_count,
        "float_count": float_count,
        "str_count": str_count,
        "agg_function": agg_function_encoded,
    }


_ = extract_features_from_dataframe(course, "course_unit", "mean")

In [ ]:
%%time
### cell 20 ###
_ = course[course["course_code"].str.startswith("CS")].value_counts()

In [ ]:
# ### cell 21 ### (cpu)

# course.sort_values("course_rating_mean", ascending=False, inplace=True)

In [ ]:
%%time
### cell 22 ###

course.reset_index(inplace=True)

In [ ]:
%%time
### cell 23 ###

course.set_index("course_unit", inplace=True)

In [ ]:
%%time
### cell 24 ###

_ = course.loc["KOREA", "course_rating_mean"].value_counts()

In [ ]:
# ### cell 25 ### (cpu)

# KOREA = course.loc["KOREA", :]

In [ ]:
%%time
### cell 26 ###

_ = course.loc["CHINA", "course_rating_mean"].value_counts()

In [ ]:
# ### cell 27 ### (cpu)
# china = course.loc["CHINA", :]

In [ ]:
%%time
### cell 28 ###

_ = course.loc["CHINA", "course_rating_mean"].value_counts()

In [ ]:
%%time
### cell 29 ###

_ = course.loc["CS", "course_rating_mean"].value_counts()

In [ ]:
### cell 30 ### (cpu)

cs = course.loc["CS", :]
# cs

In [ ]:
%%time
### cell 31 ###

_ = (
    cs.groupby("course_code")
    .mean(numeric_only=True)
    .sort_values("course_rating_int", ascending=False)
)

In [ ]:
del cs

In [ ]:
%%time
### cell 32 ###

_ = course.loc["WKRPT", "course_rating_mean"].value_counts()

In [ ]:
### cell 33 ### (cpu)

wkrpt = course.loc["WKRPT", :]
# wkrpt

In [ ]:
%%time
### cell 34 ###

_ = (
    wkrpt.groupby("course_code")
    .mean(numeric_only=True)
    .sort_values("course_rating_int", ascending=False)
)

In [ ]:
%%time
### cell 35 ###

_ = wkrpt.groupby("course_code").mean(numeric_only=True)

In [ ]:
del wkrpt

In [ ]:
%%time
### cell 36 ###

_ = course.loc["PD", "course_rating_mean"].value_counts()

In [ ]:
### cell 37 ### (cpu)

pd = course.loc["PD", :]
# pd

In [ ]:
%%time
### cell 38 ###

pd_mean = (
    pd.groupby("course_code")
    .mean(numeric_only=True)
    .sort_values("course_rating_int", ascending=False)
)

In [ ]:
del pd